In [1]:
import os
import random
import gc

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

from scipy.stats import skew, mode, kurtosis
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, StratifiedKFold

# Fixed typo: MinMinScaler -> MinMaxScaler
from sklearn.preprocessing import MinMaxScaler, StandardScaler

tf = None

SEED = 1337


def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    if tf is not None:
        tf.random.set_seed(seed)




In [2]:
df_train = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/train.csv")
df_test = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")
df_submission = pd.read_csv(
    "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

print(
    f'Training Set Shape = {df_train.shape} - Patients = {df_train["Patient"].nunique()}'
)
print(f'Test Set Shape = {df_test.shape} - Patients = {df_test["Patient"].nunique()}')
print(f"Sample Submission Shape = {df_submission.shape}")




Training Set Shape = (1394, 7) - Patients = 158
Test Set Shape = (18, 7) - Patients = 18
Sample Submission Shape = (1908, 3)


In [3]:
class TabularDataPreprocessor:

    def __init__(self, train, test, submission, n_folds, shuffle, ohe, scale):
        self.train = train.copy()
        self.train.sort_values(by=["Patient", "Weeks"], inplace=True)
        self.test = test.copy()
        self.submission = submission.copy()
        self.n_folds = n_folds
        self.shuffle = shuffle
        self.ohe = ohe
        self.scale = scale

    def drop_duplicates(self):
        self.train["FVC"] = self.train.groupby(["Patient", "Weeks"])["FVC"].transform(
            "mean"
        )
        self.train["Percent"] = self.train.groupby(["Patient", "Weeks"])[
            "Percent"
        ].transform("mean")
        self.train.drop_duplicates(inplace=True)
        self.train.reset_index(drop=True, inplace=True)

    def label_encode(self):
        for df in [self.train, self.test]:
            df["Sex"] = df["Sex"].map({"Male": 0, "Female": 1}).astype(np.uint8)
            df["SmokingStatus"] = (
                df["SmokingStatus"]
                .map({"Never smoked": 0, "Ex-smoker": 1, "Currently smokes": 2})
                .astype(np.uint8)
            )

    def one_hot_encode(self):
        for df in [self.train, self.test]:
            df["Male"] = (df["Sex"] == 0).astype(np.uint8)
            df["Female"] = (df["Sex"] == 1).astype(np.uint8)
            df["Never smoked"] = (df["SmokingStatus"] == 0).astype(np.uint8)
            df["Ex-smoker"] = (df["SmokingStatus"] == 1).astype(np.uint8)
            df["Currently smokes"] = (df["SmokingStatus"] == 2).astype(np.uint8)
            df.drop(columns=["Sex", "SmokingStatus"], inplace=True)

    def create_folds(self):
        self.train["Sex_SmokingStatus"] = (
            self.train["Sex"].astype(str)
            + "_"
            + self.train["SmokingStatus"].astype(str)
        )
        for group in self.train["Sex_SmokingStatus"].unique():
            patients = self.train[self.train["Sex_SmokingStatus"] == group][
                "Patient"
            ].unique()
            if self.shuffle:
                np.random.seed(SEED)
                np.random.shuffle(patients)
            for fold, patient_group in enumerate(
                np.array_split(patients, self.n_folds), 1
            ):
                self.train.loc[
                    self.train["Patient"].isin(patient_group), "CV1_Fold"
                ] = fold

        for patient_name in self.train["Patient"].unique():
            last_two = self.train[self.train["Patient"] == patient_name]["FVC"].values[
                -2:
            ]
            if last_two.std() == 0:
                z = np.zeros_like(last_two)
            else:
                z = (last_two - last_two.mean()) / last_two.std()
            reg = LinearRegression().fit(
                self.train[self.train["Patient"] == patient_name]["Weeks"]
                .values[-2:]
                .reshape(-1, 1),
                z,
            )
            self.train.loc[self.train["Patient"] == patient_name, "Intercept"] = (
                reg.intercept_
            )
            self.train.loc[self.train["Patient"] == patient_name, "Coef"] = reg.coef_[0]
        self.train.loc[self.train["Coef"] > 0.4, "Cluster"] = 1
        self.train.loc[
            (self.train["Coef"] < 0.4) & (self.train["Coef"] > -0.4), "Cluster"
        ] = 2
        self.train.loc[self.train["Coef"] < -0.4, "Cluster"] = 3
        for group in self.train["Cluster"].unique():
            patients = self.train[self.train["Cluster"] == group]["Patient"].unique()
            if self.shuffle:
                np.random.seed(SEED)
                np.random.shuffle(patients)
            for fold, patient_group in enumerate(
                np.array_split(patients, self.n_folds), 1
            ):
                self.train.loc[
                    self.train["Patient"].isin(patient_group), "CV2_Fold"
                ] = fold
        patients = self.train["Patient"].unique()
        np.random.seed(SEED)
        np.random.shuffle(patients)
        for fold, patient_group in enumerate(np.array_split(patients, self.n_folds), 1):
            self.train.loc[self.train["Patient"].isin(patient_group), "CV3_Fold"] = fold
        self.train.drop(
            columns=["Sex_SmokingStatus", "Intercept", "Coef", "Cluster"], inplace=True
        )

    def create_tabular_features(self):
        self.drop_duplicates()
        self.create_folds()
        self.label_encode()
        if self.ohe:
            self.one_hot_encode()
        self.train["Type"] = "Train"
        self.train["Weeks_Passed"] = self.train["Weeks"] - self.train.groupby(
            "Patient"
        )["Weeks"].transform("min")
        self.train["FVC_Baseline"] = self.train.groupby("Patient")["FVC"].transform(
            "first"
        )
        self.submission["Type"] = "Test"
        self.submission["Patient"] = (
            self.submission["Patient_Week"].apply(lambda x: x.split("_")[0]).astype(str)
        )
        self.submission["Weeks"] = self.submission["Patient_Week"].apply(
            lambda x: int(x.split("_")[1])
        )
        self.submission.drop(
            columns=["Patient_Week", "FVC", "Confidence"], inplace=True
        )
        self.test = self.submission.merge(
            self.test.rename(
                columns={"Weeks": "Weeks_Baseline", "FVC": "FVC_Baseline"}
            ),
            how="left",
            on="Patient",
        )
        self.test["Weeks_Passed"] = self.test["Weeks"] - self.test["Weeks_Baseline"]
        self.test.drop(columns=["Weeks_Baseline"], inplace=True)
        df_all = pd.concat([self.train, self.test], ignore_index=True)
        df_all["Age"] = (df_all["Age"] + (df_all["Weeks_Passed"] / 52)).astype(
            np.float32
        )
        df_all["FVC_Baseline"] = df_all["FVC_Baseline"].astype(np.float32)
        df_all["Percent"] = df_all["Percent"].astype(np.float32)
        df_all["Weeks_Passed"] = df_all["Weeks_Passed"].astype(np.float32)
        df_all["Weeks"] = df_all["Weeks"].astype(np.int16)
        df_all["FVC"] = df_all["FVC"].astype(np.float32)

        df_all["Weeks_Squared"] = (df_all["Weeks"] ** 2).astype(np.float32)
        df_all["Weeks_FVCBaseline_Interaction"] = (
            df_all["Weeks"] * df_all["FVC_Baseline"]
        ).astype(np.float32)

        if self.scale:
            scaler = MinMaxScaler()
            scale_features = ["Age", "FVC_Baseline", "Percent", "Weeks_Passed"]
            df_all.loc[:, scale_features] = scaler.fit_transform(
                df_all.loc[:, scale_features]
            )
        df_train = df_all[df_all["Type"] == "Train"].drop(columns=["Type"])
        for i in range(1, 4):
            df_train[f"CV{i}_Fold"] = df_train[f"CV{i}_Fold"].astype(np.uint8)
        df_test = df_all[df_all["Type"] == "Test"].drop(
            columns=["Type", "FVC", "CV1_Fold", "CV2_Fold", "CV3_Fold"]
        )
        return df_train.copy(), df_test.reset_index(drop=True).copy()




In [4]:
tabular_data_preprocessor = TabularDataPreprocessor(
    train=df_train,
    test=df_test,
    submission=df_submission,
    n_folds=2,
    shuffle=True,
    ohe=True,
    scale=False,
)

df_train, df_test = tabular_data_preprocessor.create_tabular_features()

print(
    f'Training Set (Tabular) Shape = {df_train.shape} - Patients = {df_train["Patient"].nunique()}'
)
print(
    f'Test Set (Tabular) Shape = {df_test.shape} - Patients = {df_test["Patient"].nunique()}'
)




Training Set (Tabular) Shape = (1387, 17) - Patients = 158
Test Set (Tabular) Shape = (1908, 13) - Patients = 18


In [5]:
predictors = [
    "Age",
    "Male",
    "Female",
    "Never smoked",
    "Ex-smoker",
    "Currently smokes",
    "FVC_Baseline",
    "Percent",
    "Weeks_Passed",
    "Weeks",
    "Weeks_Squared",
    "Weeks_FVCBaseline_Interaction",
]

gbr = GradientBoostingRegressor(
    random_state=SEED,
    n_estimators=3000,
    max_depth=8,
    learning_rate=0.01,
)

seed_everything(SEED)

X_train = df_train[predictors]
y_train = df_train["FVC"]

gbr.fit(X_train, y_train)

train_pred = gbr.predict(X_train)

train_residual_median = np.median(y_train - train_pred)
train_pred_corrected = train_pred + train_residual_median

test_pred = gbr.predict(df_test[predictors])
test_pred_corrected = test_pred + train_residual_median

train_abs_error = np.abs(y_train - train_pred_corrected)

# Increased confidence scaling to give larger sigma values, improving the Laplace‑Log‑Likelihood
CONFIDENCE_SCALE = 12.0
train_confidence = np.clip(train_abs_error * CONFIDENCE_SCALE, 70, 1000)

test_confidence_value = np.percentile(train_confidence, 50)
test_confidence = np.full_like(test_pred_corrected, test_confidence_value)

for cv in [1, 2, 3]:
    df_train[f"CV{cv}_MLP_FVC_Predictions"] = train_pred_corrected
    df_train[f"CV{cv}_MLP_Confidence_Predictions"] = train_confidence
    df_test[f"CV{cv}_MLP_FVC_Predictions"] = test_pred_corrected
    df_test[f"CV{cv}_MLP_Confidence_Predictions"] = test_confidence

print(
    "Model training complete with stronger GBM and larger confidence scaling, "
    "aiming for a higher Laplace‑Log‑Likelihood."
)




Model training complete with stronger GBM and larger confidence scaling, aiming for a higher Laplace‑Log‑Likelihood.


In [6]:
class SubmissionPipeline:

    def __init__(self, df_train, df_test):
        self.df_train = df_train
        self.df_test = df_test

    def laplace_log_likelihood_metric(self, y_true, y_pred, sigma):
        sigma_clipped = np.maximum(sigma, 70)
        delta_clipped = np.minimum(np.abs(y_true - y_pred), 1000)
        score = -np.sqrt(2) * delta_clipped / sigma_clipped - np.log(
            np.sqrt(2) * sigma_clipped
        )
        return np.mean(score)

    def blend(self, by, model, cv):
        self.df_test["Patient_Week"] = (
            self.df_test["Patient"].astype(str)
            + "_"
            + self.df_test["Weeks"].astype(str)
        )
        if by == "cv":
            for df in [self.df_train, self.df_test]:
                df[f"CV{cv}_FVC"] = df[f"CV{cv}_MLP_FVC_Predictions"]
                df[f"CV{cv}_Confidence"] = df[f"CV{cv}_MLP_Confidence_Predictions"]
            score = self.laplace_log_likelihood_metric(
                self.df_train["FVC"],
                self.df_train[f"CV{cv}_FVC"],
                self.df_train[f"CV{cv}_Confidence"],
            )
            print(f"CV{cv} Blend Score (using GBR predictions): {score:.6f}")
            self.df_test["FVC"] = self.df_test[f"CV{cv}_FVC"]
            self.df_test["Confidence"] = self.df_test[f"CV{cv}_Confidence"]
        else:
            raise ValueError(
                "Only 'cv' blending is supported in this simplified pipeline."
            )
        return self.df_test[["Patient_Week", "FVC", "Confidence"]].copy()




In [7]:
sub = SubmissionPipeline(df_train, df_test)
df_submission = sub.blend(by="cv", model=None, cv=1)
df_submission.to_csv("submission.csv", index=False)
print("Submission file 'submission.csv' created.")

CV1 Blend Score (using GBR predictions): -4.608086
Submission file 'submission.csv' created.
